<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_01_tensor_shapes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-01: Tensor Shapes & Reshaping

**Difficulty**: ⬜ Warm-up

**Objective**: Master tensor creation and shape manipulation — the single most common operation in all of deep learning.

## What you'll practice
- `view`, `reshape`, `transpose`, `permute`
- `squeeze`, `unsqueeze`, `flatten`
- `contiguous()` and when it matters
- `expand` vs `repeat`

## Rules
- Use only basic PyTorch tensor ops
- No numpy
- Read each task, predict the answer, THEN run it

In [1]:
import torch
import torch.nn.functional as F

## Task 1: Predict & Verify Shapes

For each operation, **write your predicted shape as a comment BEFORE running the cell**.

In [14]:
x = torch.randn(2, 3, 4)

# TODO: Predict each shape before running
a = x.view(6, 4)          # predicted shape: ???
b = x.view(2, -1)         # predicted shape: ???
c = x.reshape(24)         # predicted shape: ???
d = x.flatten()           # predicted shape: ???
e = x.flatten(1)          # predicted shape: ???  (flatten starting from dim 1)

print(f"a: {a.shape}")
print(f"b: {b.shape}")
print(f"c: {c.shape}")
print(f"d: {d.shape}")
print(f"e: {e.shape}")

a: torch.Size([6, 4])
b: torch.Size([2, 12])
c: torch.Size([24])
d: torch.Size([24])
e: torch.Size([2, 12])


## Task 2: Transpose & Permute

A common attention pattern: rearrange `(batch, seq, heads, dim)` → `(batch, heads, seq, dim)`

In [15]:
# Simulate multi-head attention input: (batch=2, seq_len=10, num_heads=8, head_dim=64)
x = torch.randn(2, 10, 8, 64)

# TODO: Rearrange to (batch, num_heads, seq_len, head_dim) using .transpose()
# YOUR CODE HERE
y_transpose = x.transpose(1, 2)

# TODO: Same thing using .permute()
# YOUR CODE HERE
y_permute = x.permute(0, 2, 1, 3)

# Verify
assert y_transpose.shape == (2, 8, 10, 64), f"Got {y_transpose.shape}"
assert y_permute.shape == (2, 8, 10, 64), f"Got {y_permute.shape}"
assert torch.equal(y_transpose, y_permute)
print("✅ Task 2 passed!")

✅ Task 2 passed!


## Task 3: Squeeze & Unsqueeze

Adding/removing dimensions of size 1. Critical for broadcasting.

In [31]:
x = torch.randn(3, 1, 4)

# TODO: Remove the dimension of size 1
squeezed = x.squeeze(1)  # YOUR CODE HERE — should be shape (3, 4)

# TODO: Add a dimension at position 0 to make it (1, 3, 4)
unsqueezed = x.squeeze(1).unsqueeze(0)  # YOUR CODE HERE

# TODO: Given a (5,) vector, make it (1, 5) and (5, 1) using unsqueeze
v = torch.randn(5)
row = v.unsqueeze(0)  # YOUR CODE — shape (1, 5)
col = v.unsqueeze(1)  # YOUR CODE — shape (5, 1)

assert squeezed.shape == (3, 4)
assert unsqueezed.shape == (1, 3, 4)
assert row.shape == (1, 5)
assert col.shape == (5, 1)
print("✅ Task 3 passed!")

✅ Task 3 passed!


## Task 4: Contiguous & View vs Reshape

In [36]:
x = torch.randn(3, 4)
y = x.T  # transpose makes it non-contiguous

print(f"x is contiguous: {x.is_contiguous()}")
print(f"y is contiguous: {y.is_contiguous()}")

# TODO: Try y.view(12) — what happens? Catch the error.
try:
    z = y.view(12)
    print("view worked")
except RuntimeError as e:
    print(f"view failed: {e}")

# TODO: Fix it two ways:
# Way 1: Make contiguous first, then view
z1 = y.contiguous().view(12)  # YOUR CODE HERE

# Way 2: Use reshape (handles non-contiguous automatically)
z2 = y.reshape(12)  # YOUR CODE HERE

assert z1.shape == (12,)
assert z2.shape == (12,)
print("✅ Task 4 passed!")

x is contiguous: True
y is contiguous: False
view failed: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
✅ Task 4 passed!


## Task 5: Expand vs Repeat

`expand` = broadcast view (no memory copy), `repeat` = actual copy.

In [43]:
# A bias vector (1, 4) that we want to apply to a (3, 4) tensor
bias = torch.randn(1, 4)

# TODO: Use expand to make it (3, 4) without copying memory
expanded = bias.expand(3, 4)  # YOUR CODE HERE

# TODO: Use repeat to make it (3, 4) with an actual copy
repeated = bias.repeat(3, 1)  # YOUR CODE HERE

assert expanded.shape == (3, 4)
assert repeated.shape == (3, 4)

# Key difference: expand shares memory, repeat doesn't
print(f"expanded shares storage: {expanded.data_ptr() == bias.data_ptr()}")
print(f"repeated shares storage: {repeated.data_ptr() == bias.data_ptr()}")
print("✅ Task 5 passed!")

expanded shares storage: True
repeated shares storage: False
✅ Task 5 passed!


## Task 6: Cat vs Stack

Both join tensors, but `stack` creates a NEW dimension.

In [48]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)

# TODO: Concatenate along dim 0 → shape (6, 4)
cat_0 = torch.concatenate([a, b], dim=0)  # YOUR CODE

# TODO: Concatenate along dim 1 → shape (3, 8)
cat_1 = torch.concatenate([a, b], dim=1)  # YOUR CODE

# TODO: Stack along dim 0 → shape (2, 3, 4)
stacked = torch.stack([a, b], dim=0)  # YOUR CODE

assert cat_0.shape == (6, 4)
assert cat_1.shape == (3, 8)
assert stacked.shape == (2, 3, 4)
print("✅ Task 6 passed!")

✅ Task 6 passed!


## Task 7: Multi-Head Split & Merge (Attention Pattern)

This exact pattern appears in EVERY attention implementation.

In [60]:
B, N, D = 2, 10, 512  # batch, seq_len, model_dim
num_heads = 8
head_dim = D // num_heads  # 64

x = torch.randn(B, N, D)

# TODO: Split into heads: (B, N, D) → (B, num_heads, N, head_dim)
# Step 1: reshape to (B, N, num_heads, head_dim)
# Step 2: transpose to (B, num_heads, N, head_dim)
multi_head = x.reshape(B, N, num_heads, head_dim).transpose(2, 1)  # YOUR CODE HERE

assert multi_head.shape == (B, num_heads, N, head_dim)

# TODO: Merge heads back: (B, num_heads, N, head_dim) → (B, N, D)
# Step 1: transpose to (B, N, num_heads, head_dim)
# Step 2: reshape to (B, N, D) — use .contiguous() before .view()!
merged = multi_head.transpose(2, 1).reshape(B, N, D)  # YOUR CODE HERE

assert merged.shape == (B, N, D)
assert torch.allclose(x, merged)  # should be identical
print("✅ Task 7 passed! You now know THE attention reshape pattern.")

✅ Task 7 passed! You now know THE attention reshape pattern.
